In [19]:
import time
import requests
import pandas as pd
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'

df = pd.read_csv(DATA_DIR / 'final_data_v2.csv', low_memory=False)
df['college_id'] = df['college_id'].astype(str).str.strip()
df['cycle'] = pd.to_numeric(df['cycle'], errors='coerce').astype('Int64')
print('final_data_v2 shape:', df.shape)
df.head(2)

final_data_v2 shape: (1048575, 41)


,college_id,cycle,school_type,hs_id,hs_name,col_name,col_city,col_st,col_zip,col_type,...,hs_pct_two_or_more,hs_title_i_status,hs_urban_centric_locale,hs_charter,hs_magnet,hs_enrollment,hs_pct_free_or_reduced_price_lunch,hs_students_per_teacher,hs_school_level,hs_highest_grade_offered
0,45896401,2022,public,450201000242,Gregg Middle,STRAYER UNIVERSITY - CHARLESTON CAMPUS,NORTH CHARLESTON,SC,29418.0,3.0,...,0.088235,NaN,21.0,0.0,NaN,850.0,0.757647,14.655172,middle,8.0
1,101189,2023,private,A1900014,EASTWOOD CHRISTIAN SCHOOL,FAULKNER UNIVERSITY,MONTGOMERY,AL,36109.0,2.0,...,0.032389,-10.0,-10.0,-10.0,-10.0,247.0,-10.000000,12.350000,combined,12.0


In [20]:
# Shared HTTP session with retries
session = requests.Session()
retries = Retry(
    total=8, connect=8, read=8, status=8,
    backoff_factor=0.75,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(['GET']),
    respect_retry_after_header=True,
)
session.mount('https://', HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10))
session.headers.update({'User-Agent': 'c2i-ipeds/1.0'})

def fetch_all(url):
    rows = []
    while url:
        for attempt in range(6):
            try:
                resp = session.get(url, timeout=(15, 120))
                resp.raise_for_status()
                data = resp.json()
                break
            except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError):
                if attempt == 5:
                    raise
                time.sleep(1.5 * (attempt + 1))
        rows.extend(data.get('results', []))
        url = data.get('next')
        time.sleep(0.05)
    return rows

In [21]:
# Pull admissions/enrollment data (same approach as geo.ipynb)
# Data available 2019-2022; 2022 data will be forward-filled to 2023
# Filter sex==99 (total across all sexes)
admissions_years = [2019, 2020, 2021, 2022]
print('Pulling admissions for years:', admissions_years)

all_admissions = []
for year in admissions_years:
    url = f'https://educationdata.urban.org/api/v1/college-university/ipeds/admissions-enrollment/{year}/'
    rows = fetch_all(url)
    all_admissions.extend(rows)
    print(f'  {year}: {len(rows):,} rows')

admissions_df = pd.DataFrame(all_admissions)
admissions_df['sex'] = pd.to_numeric(admissions_df['sex'], errors='coerce')
admissions_df = admissions_df[admissions_df['sex'] == 99].copy()

college_admissions = admissions_df[['unitid', 'year', 'number_applied', 'number_admitted', 'number_enrolled_total']].copy()
college_admissions.rename(columns={'unitid': 'college_id', 'year': 'cycle'}, inplace=True)
college_admissions['college_id'] = college_admissions['college_id'].astype(str).str.strip()
college_admissions['cycle'] = pd.to_numeric(college_admissions['cycle'], errors='coerce').astype('Int64')

college_admissions['col_acceptance_rate'] = (
    college_admissions['number_admitted'] / college_admissions['number_applied']
)
college_admissions['col_enrollment_rate'] = (
    college_admissions['number_enrolled_total'] / college_admissions['number_admitted']
)

college_admissions.rename(columns={
    'number_applied':        'col_number_applied',
    'number_admitted':       'col_number_admitted',
    'number_enrolled_total': 'col_number_enrolled_total',
}, inplace=True)

college_admissions = college_admissions.drop_duplicates(subset=['college_id', 'cycle'])

# Forward-fill 2022 -> 2023
admissions_2022 = college_admissions[college_admissions['cycle'] == 2022].copy()
admissions_2022['cycle'] = pd.array([2023], dtype='Int64')[0]
college_admissions = pd.concat([college_admissions, admissions_2022], ignore_index=True)

print('college_admissions shape (with 2023 fill):', college_admissions.shape)
college_admissions.head()

Pulling admissions for years: [2019, 2020, 2021, 2022]
  2019: 6,033 rows
  2020: 5,967 rows
  2021: 5,943 rows
  2022: 8,686 rows
college_admissions shape (with 2023 fill): (9963, 7)


,college_id,cycle,col_number_applied,col_number_admitted,col_number_enrolled_total,col_acceptance_rate,col_enrollment_rate
0,100654,2019,9579.0,8789.0,1710.0,0.917528,0.194561
1,100663,2019,8298.0,6112.0,2346.0,0.736563,0.383835
2,100706,2019,5295.0,4372.0,1497.0,0.825685,0.342406
3,100724,2019,6674.0,6467.0,1023.0,0.968984,0.158188
4,100751,2019,38505.0,31835.0,6764.0,0.826776,0.212471


In [22]:
# Pull tuition data from IPEDS academic-year-tuition endpoint
# Data available 2019-2021; 2021 data will be forward-filled to 2022 and 2023
tuition_years = [2019, 2020, 2021]
print('Pulling tuition for years:', tuition_years)

all_tuition = []
for year in tuition_years:
    url = f'https://educationdata.urban.org/api/v1/college-university/ipeds/academic-year-tuition/{year}/'
    rows = fetch_all(url)
    all_tuition.extend(rows)
    print(f'  {year}: {len(rows):,} rows')

tuition_df = pd.DataFrame(all_tuition)
print('Raw tuition columns:', tuition_df.columns.tolist())
tuition_df.head(2)

Pulling tuition for years: [2019, 2020, 2021]
  2019: 17,296 rows
  2020: 16,948 rows
  2021: 16,858 rows
Raw tuition columns: ['unitid', 'year', 'fips', 'level_of_study', 'tuition_type', 'tuition_fees_ft', 'tuition_ft', 'fees_ft', 'credit_hour_charge_pt', 'tuition_published', 'fees_published', 'tuition_fees_published', 'tuition_pub_pct_increase', 'fees_pub_pct_increase']


,unitid,year,fips,level_of_study,tuition_type,tuition_fees_ft,tuition_ft,fees_ft,credit_hour_charge_pt,tuition_published,fees_published,tuition_fees_published,tuition_pub_pct_increase,fees_pub_pct_increase
0,100654,2019,1,1,2,10024.0,8610.0,1414.0,287.0,8610.0,1414.0,10024.0,NaN,NaN
1,100654,2019,1,1,3,10024.0,8610.0,1414.0,287.0,8610.0,1414.0,10024.0,NaN,NaN


In [23]:
# Keep relevant columns and rename
# Tuition endpoint may segment by in-state/out-of-state and student level;
# inspect unique values to pick the right filter
print(tuition_df.dtypes)
for col in ['level_of_study', 'tuition_type', 'state_tuition']:
    if col in tuition_df.columns:
        print(f'\n{col} unique values:', tuition_df[col].unique())

unitid                        int64
year                          int64
fips                          int64
level_of_study                int64
tuition_type                  int64
tuition_fees_ft             float64
tuition_ft                  float64
fees_ft                     float64
credit_hour_charge_pt       float64
tuition_published           float64
fees_published              float64
tuition_fees_published      float64
tuition_pub_pct_increase    float64
fees_pub_pct_increase       float64
dtype: object

level_of_study unique values: [1 2]

tuition_type unique values: [2 3 4]


In [24]:
# Filter to undergraduate (level_of_study == 1) and aggregate published tuition
keep = tuition_df.copy()
if 'level_of_study' in keep.columns:
    keep = keep[pd.to_numeric(keep['level_of_study'], errors='coerce') == 1]

keep = keep[['unitid', 'year', 'tuition_published', 'tuition_fees_ft']].copy()
keep['unitid'] = keep['unitid'].astype(str).str.strip()
keep['year'] = pd.to_numeric(keep['year'], errors='coerce').astype('Int64')
for col in ['tuition_published', 'tuition_fees_ft']:
    keep[col] = pd.to_numeric(keep[col], errors='coerce')

# If multiple rows per (unitid, year) remain, take the max (out-of-state)
college_tuition = (
    keep.groupby(['unitid', 'year'], as_index=False)
        .agg({'tuition_published': 'max', 'tuition_fees_ft': 'max'})
)
college_tuition.rename(columns={
    'unitid': 'college_id',
    'year':   'cycle',
    'tuition_published': 'col_tuition_published',
    'tuition_fees_ft':   'col_tuition_fees_ft',
}, inplace=True)

college_tuition = college_tuition.drop_duplicates(subset=['college_id', 'cycle'])

# Forward-fill 2021 -> 2022 and 2023
tuition_2021 = college_tuition[college_tuition['cycle'] == 2021].copy()
tuition_2022 = tuition_2021.copy(); tuition_2022['cycle'] = pd.array([2022], dtype='Int64')[0]
tuition_2023 = tuition_2021.copy(); tuition_2023['cycle'] = pd.array([2023], dtype='Int64')[0]
college_tuition = pd.concat([college_tuition, tuition_2022, tuition_2023], ignore_index=True)

print('college_tuition shape (with 2022/2023 fill):', college_tuition.shape)
college_tuition.head()

college_tuition shape (with 2022/2023 fill): (18097, 4)


,college_id,cycle,col_tuition_published,col_tuition_fees_ft
0,100654,2019,17220.0,18634.0
1,100654,2020,17220.0,18634.0
2,100654,2021,17220.0,18634.0
3,100663,2019,20400.0,20400.0
4,100663,2020,20400.0,20400.0


In [25]:
# Merge admissions and tuition onto main dataset
df_out = df.merge(college_admissions, on=['college_id', 'cycle'], how='left')
df_out = df_out.merge(college_tuition,  on=['college_id', 'cycle'], how='left')

assert len(df_out) == len(df), f'Row count changed: {len(df)} -> {len(df_out)}'

new_cols = ['col_number_applied', 'col_number_admitted', 'col_number_enrolled_total',
            'col_acceptance_rate', 'col_enrollment_rate',
            'col_tuition_published', 'col_tuition_fees_ft']

print('Fill rates for new columns:')
print((df_out[new_cols].notna().mean() * 100).round(1))
print('\nFinal shape:', df_out.shape)
df_out.head(2)

Fill rates for new columns:
col_number_applied           74.2
col_number_admitted          74.1
col_number_enrolled_total    74.1
col_acceptance_rate          74.1
col_enrollment_rate          74.1
col_tuition_published        78.2
col_tuition_fees_ft          80.9
dtype: float64

Final shape: (1048575, 48)


,college_id,cycle,school_type,hs_id,hs_name,col_name,col_city,col_st,col_zip,col_type,...,hs_students_per_teacher,hs_school_level,hs_highest_grade_offered,col_number_applied,col_number_admitted,col_number_enrolled_total,col_acceptance_rate,col_enrollment_rate,col_tuition_published,col_tuition_fees_ft
0,45896401,2022,public,450201000242,Gregg Middle,STRAYER UNIVERSITY - CHARLESTON CAMPUS,NORTH CHARLESTON,SC,29418.0,3.0,...,14.655172,middle,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,101189,2023,private,A1900014,EASTWOOD CHRISTIAN SCHOOL,FAULKNER UNIVERSITY,MONTGOMERY,AL,36109.0,2.0,...,12.350000,combined,12.0,1728.0,1424.0,238.0,0.824074,0.167135,21500.0,23490.0


In [26]:
out_path = DATA_DIR / 'final_data_v3.csv'
df_out.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

Saved: /Users/bryceclement/Desktop/c2i/data/final_data_v3.csv
